# 5G Network Intelligence — Available Target Supervision

## Research question and hypothesis
Can retaining valid short-horizon fit labels improve forecasts? Compare newly trained joint-label and available-label arms with the same architecture, seed and horizon-balanced training objective. The old control is not silently treated as loss-matched. Implementation tests establish causality, not an accuracy gain.


In [ ]:
# Review Run All never trains. A separate hosted worker enables this flag.
RUN_TRAINING = False


## Frozen temporal protocol
Two reused development folds: fit/stop/score endpoints 335/383/443 and 395/443/503. Context 168h, all 2,880 beams, horizons 1/6/24h. Original hours 504 onward remain excluded. Both arms keep complete joint stopping windows and identical full per-horizon scoring populations. No final refit, holdout tuning or production promotion.


In [ ]:
from pathlib import Path, PurePosixPath
import os, sys, json, hashlib, shutil
PROJECT = Path.cwd()
if not (PROJECT/'configs').is_dir():
    PROJECT = PROJECT.parent
def verified_input(filename, digest):
    matches = [p for p in Path('/kaggle/input').rglob(filename)
               if hashlib.sha256(p.read_bytes()).hexdigest() == digest]
    if len(matches) != 1:
        raise ValueError('Exactly one pinned input manifest required')
    source = matches[0]
    manifest = json.loads(source.read_text())
    for name, expected in manifest['files'].items():
        relative = PurePosixPath(name)
        if relative.is_absolute() or '..' in relative.parts or ':' in name or '\\' in name:
            raise ValueError('Unsafe input path')
        path = source.parent/name
        if not path.resolve().is_relative_to(source.parent.resolve()) or path.is_symlink():
            raise ValueError('Input path escapes manifest root')
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise ValueError('Input checksum differs: '+name)
    return source.parent, manifest


## Input provenance
The new private supplement contains only MIT code/config. Reuse original observations from the already approved hosted input after manifest verification. No local data export, external pretraining corpus or checkpoint transfer is needed.


In [ ]:
if RUN_TRAINING:
    assert Path('/kaggle/working').is_dir(), 'Hosted Kaggle only.'
    base_root, base = verified_input('source_manifest.json',
        '3069c57c9f64f809464ead2fae973fbaa35a86c3a46884e15cc693354eecab46')
    code_root, code = verified_input('partial_supervision_source_manifest.json', 'f4c54ca989621a1ef80d89d57573c31ada7a5363e664e91876dc8d65f1dc0dac')
    assert not code['data_or_checkpoint_included']
    PROJECT = Path('/kaggle/working/partial_supervision_study')
    PROJECT.mkdir(exist_ok=True)
    # Reuse only canonical original panel/split. External source corpora are unused.
    for name in ['data/processed/panel.npz', 'data/processed/panel_meta.json', 'data/processed/split.json']:
        target = PROJECT/name
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(base_root/name, target)
    for name in code['files']:
        target = PROJECT/name
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(code_root/name, target)
    os.chdir(PROJECT)
    sys.path.insert(0, str(PROJECT))
    os.environ['SOURCE_GIT_COMMIT'] = code['source_commit']
    print('Verified training source:', code['source_commit'])


## Configuration
Both arms use seed 17, cell attention, maximum 100 epochs and patience 12. Global per-KPI medians/scales fit only the training prefix; phase4 and missing masks are retained.


In [ ]:
import yaml
import pandas as pd
from IPython.display import display
settings = []
for arm in ['joint', 'available']:
    path = PROJECT/'configs'/f'partial_supervision_{arm}.yaml'
    if path.exists():
        cfg = yaml.safe_load(path.read_text(encoding='utf-8'))
        settings.append({'arm': arm, 'seed': cfg['seed'], 'batch_size': cfg['batch_size'],
            'max_epochs': cfg['max_epochs'], 'patience': cfg['early_stopping_patience'],
            'training_objective': cfg['training_objective'], 'stopping': cfg['stopping_label_policy']})
display(pd.DataFrame(settings))


## Causal labels and loss
Crop the backing tensors before transformation/gathering. An origin+horizon beyond the fit cutoff is NaN, never an imputed target. For observed labels, average MAE within each horizon and then across supervised horizons in each batch. Use this same loss in both arms. With complete observed prefixes, available labels add 118,080 targets per fold (9.43%/6.67%). Natural window counts and optimizer updates also increase; do not call this a pure data effect without a step-matched control.


In [ ]:
if RUN_TRAINING:
    import subprocess
    (PROJECT/'reports').mkdir(exist_ok=True)
    with (PROJECT/'reports/partial_supervision_training.log').open('w') as log:
        process = subprocess.Popen([sys.executable, '-u', '-m', 'scripts.train_partial_supervision'],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        TRAIN_EXIT = process.wait()
    if TRAIN_EXIT:
        raise RuntimeError('Inspect durable logs/checkpoints before any repeat fit.')
else:
    print('Review mode: no fitting.')


## Debugging and independent evaluation
Save per-horizon supervision counts, cold initialization, best/last checkpoints, optimizer/RNG and stopping history in Kaggle Output. Training logs contain a weighted average of batch objectives, not a reconstructed global macro MAE. Stopping logs use physical DLPRB MAE despite the historical `scaled` field name. Verify the loss, counts, axes, preprocessing and numerical parity before assessing quality. MAE uses DLPRB usage units; WAPE is total absolute error divided by total absolute observed load.


In [ ]:
# Inspect completed results only; no new selection or calibration in review.
for arm in ['joint', 'available']:
    path = PROJECT/'reports'/f'partial_supervision_{arm}_development.csv'
    if path.exists():
        frame = pd.read_csv(path)
        frame['WAPE (%)'] = 100*frame['wape']
        print(arm)
        display(frame)


## Decision and limitations
Require a full-population comparison with the new joint control and fixed strong blend, report every fold/horizon and negative result. A 2% mean MAE point gain without regression is an initial screen; a 5% material gain still needs robustness and uncertainty evidence. WAPE below 10% is unproven. More labels do not guarantee lower error; training exposure and per-batch loss weighting must be disclosed. This is reused development, not fresh generalization. The existing Telecom study is not changed or retrained by this guide.
